In [2]:
import os
import pandas as pd
import warnings

In [4]:
# Suppress pandas indexing and slicing copy warnings from the console completely
warnings.filterwarnings('ignore', category=UserWarning)

In [5]:
# Define input filenames
GENE_POSITIONS_FILE = "compiled_gene_positions.csv"
MASTER_DATA_FILE = "IIT jammu/DataSet_variants.tsv"
CHUNK_SIZE = 250000  # Large chunk size for optimal offline processing speed


In [6]:
print("==================================================================")
print("  STEP 1: LOADING GENE LOOKUP DICTIONARY FROM CSV                ")
print("==================================================================")

# 1. Read structural targets from the provided CSV file
gene_df = pd.read_csv(GENE_POSITIONS_FILE)
gene_map = {}

for _, row in gene_df.iterrows():
    gene_name = str(row["Gene Name"]).strip()
    
    # Clean up chromosome syntax representation (e.g., turn '3.0' or ' 3 ' into 'chr3')
    chrom = str(row["Chromosome No."]).split('.')[0].strip().lower()
    if not chrom.startswith("chr"):
        chrom = f"chr{chrom}"
        
    pos_string = str(row["Genomic Position (Start to End)"])
    start_bp, end_bp = map(int, pos_string.lower().split(" to "))
    
    # Initialize chromosome keys inside our fast-lookup map
    if chrom not in gene_map:
        gene_map[chrom] = []
        
    gene_map[chrom].append({
        "name": gene_name,
        "start": start_bp,
        "end": end_bp,
        "out_file": f"{gene_name}_variants.vcf",
        "initialized": False
    })

print("Successfully mapped tracking intervals into memory:")
for ch, g_list in gene_map.items():
    print(f"  -> {ch}: Managing {len(g_list)} target region(s)")

  STEP 1: LOADING GENE LOOKUP DICTIONARY FROM CSV                
Successfully mapped tracking intervals into memory:
  -> chr3: Managing 5 target region(s)
  -> chr19: Managing 5 target region(s)
  -> chr13: Managing 2 target region(s)
  -> chr12: Managing 2 target region(s)
  -> chrx: Managing 2 target region(s)
  -> chr1: Managing 2 target region(s)
  -> chr9: Managing 2 target region(s)
  -> chr4: Managing 4 target region(s)
  -> chr10: Managing 4 target region(s)
  -> chr2: Managing 5 target region(s)
  -> chr7: Managing 3 target region(s)
  -> chr11: Managing 2 target region(s)
  -> chr17: Managing 3 target region(s)
  -> chr8: Managing 2 target region(s)


In [7]:
print("\n==================================================================")
print("  STEP 2: STREAMING MASTER VARIANT DATASET Offline                 ")
print("==================================================================")

chunk_count = 0
total_hits_saved = 0

# 2. Iterate through the 9 GB master dataset chunk-by-chunk
for chunk in pd.read_csv(MASTER_DATA_FILE, sep="\t", chunksize=CHUNK_SIZE, low_memory=False):
    chunk_count += 1
    
    # Safely cast indexing parameters into standardized data formats
    chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip().str.lower()
    chunk.iloc[:, 1] = pd.to_numeric(chunk.iloc[:, 1], errors="coerce")
    
    # Drop rows that have completely broken or corrupted coordinate positions
    valid_chunk = chunk.dropna(subset=[chunk.columns[1]])
    
    # Process the valid rows inside the current data frame matrix block
    for idx, row in valid_chunk.iterrows():
        row_chrom = row.iloc[0]
        row_pos = int(row.iloc[1])
        
        # Performance fast-path shortcut: Only check positions if chromosome matches targets
        if row_chrom in gene_map:
            for gene in gene_map[row_chrom]:
                if gene["start"] <= row_pos <= gene["end"]:
                    total_hits_saved += 1
                    
                    # Extract variables safely by explicit integer index locations
                    # This prevents failure if column headers vary slightly across versions
                    alt_allele = str(row.iloc[2]).strip()
                    ref_allele = str(row.iloc[3]).strip()
                    
                    # Explicitly catch missing float data (NaN) and replace with standard VCF dot '.'
                    beta_val = str(row.iloc[4]).strip() if pd.notna(row.iloc[4]) else "."
                    eaf_val = str(row.iloc[6]).strip() if pd.notna(row.iloc[6]) else "."
                    
                    # Strip 'chr' prefix for standard VCF body rows (e.g., 'chr3' -> '3')
                    vcf_chrom = row_chrom.replace("chr", "")
                    variant_id = f"{gene['name'].lower()}_{total_hits_saved}"
                    info_field = f"EAF={eaf_val};BETA={beta_val}"
                    
                    # Construct a perfectly formatted, tab-delimited VCF line string
                    vcf_line = f"{vcf_chrom}\t{row_pos}\t{variant_id}\t{ref_allele}\t{alt_allele}\t.\t.\t{info_field}\n"
                    
                    # File I/O handler block
                    if not gene["initialized"]:
                        # If first hit for the gene, write out the official metadata layout
                        with open(gene["out_file"], "w", encoding="utf-8") as f:
                            f.write("##fileformat=VCFv4.2\n")
                            f.write("##assembly=GRCh37\n")
                            f.write("##INFO=<ID=EAF,Number=1,Type=Float,Description=\"Effect Allele Frequency\">\n")
                            f.write("##INFO=<ID=BETA,Number=1,Type=Float,Description=\"Beta Effect Size\">\n")
                            f.write("#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n")
                            f.write(vcf_line)
                        gene["initialized"] = True
                        print(f" [VCF INITIALIZED] -> Located target variant for {gene['name']}! Created {gene['out_file']}")
                    else:
                        # Otherwise, smoothly append line to bottom of file
                        with open(gene["out_file"], "a", encoding="utf-8") as f:
                            f.write(vcf_line)
                            
    # Terminal progress tracker update loop
    if chunk_count % 20 == 0:
        processed_millions = (chunk_count * CHUNK_SIZE) / 1000000
        print(f"  Processed {processed_millions:.1f} Million rows... (Current saved hit count: {total_hits_saved})")

print("\n==================================================================")
print(f" SUCCESS! Extraction finished. Total VCF variants isolated: {total_hits_saved}")
print("==================================================================")


  STEP 2: STREAMING MASTER VARIANT DATASET Offline                 


C:\Users\lenovo\AppData\Local\Temp\ipykernel_16344\3892116483.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0         1
1         1
2         1
3         1
4         1
         ..
249995    1
249996    1
249997    1
249998    1
249999    1
Name: chromosome, Length: 250000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip().str.lower()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_16344\3892116483.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '250000    1
250001    1
250002    1
250003    1
250004    1
         ..
499995    1
499996    1
499997    1
499998    1
499999    1
Name: chromosome, Length: 250000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 

  Processed 5.0 Million rows... (Current saved hit count: 0)


C:\Users\lenovo\AppData\Local\Temp\ipykernel_16344\3892116483.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '5000000    1
5000001    1
5000002    1
5000003    1
5000004    1
          ..
5249995    1
5249996    1
5249997    1
5249998    1
5249999    1
Name: chromosome, Length: 250000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip().str.lower()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_16344\3892116483.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '5250000    1
5250001    1
5250002    1
5250003    1
5250004    1
          ..
5499995    1
5499996    1
5499997    1
5499998    1
5499999    1
Name: chromosome, Length: 250000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype f

  Processed 10.0 Million rows... (Current saved hit count: 0)


...

  Processed 65.0 Million rows... (Current saved hit count: 0)


C:\Users\lenovo\AppData\Local\Temp\ipykernel_16344\3892116483.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '65000000    2
65000001    2
65000002    2
65000003    2
65000004    2
           ..
65249995    2
65249996    2
65249997    2
65249998    2
65249999    2
Name: chromosome, Length: 250000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip().str.lower()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_16344\3892116483.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '65250000    2
65250001    2
65250002    2
65250003    2
65250004    2
           ..
65499995    2
65499996    2
65499997    2
65499998    2
65499999    2
Name: chromosome, Length: 250000, dtype: object' has dtype incompatible with int64, please explicitly cast t

  Processed 70.0 Million rows... (Current saved hit count: 0)


C:\Users\lenovo\AppData\Local\Temp\ipykernel_16344\3892116483.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '70000000    2
70000001    2
70000002    2
70000003    2
70000004    2
           ..
70249995    2
70249996    2
70249997    2
70249998    2
70249999    2
Name: chromosome, Length: 250000, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  chunk.iloc[:, 0] = chunk.iloc[:, 0].astype(str).str.strip().str.lower()
C:\Users\lenovo\AppData\Local\Temp\ipykernel_16344\3892116483.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '70250000    2
70250001    2
70250002    2
70250003    2
70250004    2
           ..
70499995    2
70499996    2
70499997    2
70499998    2
70499999    2
Name: chromosome, Length: 250000, dtype: object' has dtype incompatible with int64, please explicitly cast t

KeyboardInterrupt: 